# Name: M Danish Zaheer Awan
# Roll no: 25280092

# Data Cleaning

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

## Folders Path Defined

In [2]:
root = Path.cwd().resolve().parent.parent
folderpath = root / "Part-1"
data = root / "Data"
cleaned_data = data / "cleaned_data"
cleaned_data.mkdir(exist_ok=True)

## Synthetic Data Cleaning

In [3]:
df = pd.read_csv(data / "corrupted_synthetic_dataset.csv", low_memory=False)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
display(df.head())

print("\nNull counts:")
display(df.isnull().sum().sort_values(ascending=False))

if "sale_id" in df.columns:
    print("Duplicate sale_id count:", df["sale_id"].duplicated().sum())

print("\nDtypes:")
print(df.dtypes)

Shape: (1040203, 9)

Columns:
['sale_id', 'customer_id', 'product_id', 'sale_ts', 'quantity', 'amount', 'channel', 'currency_raw', 'notes']


,sale_id,customer_id,product_id,sale_ts,quantity,amount,channel,currency_raw,notes
0,2,93717.0,10536.0,2026-02-21 07:55:42.204402+05,2,27.84,store,EUR,NaN
1,3,140749.0,4467.0,2025-05-16 11:12:42.204402+05,9,180.83,online,GBP,NaN
2,5,1957.0,1721.0,2026-01-08 07:52:42.204402+05,3,209.91,mobile_app,GBP,NaN
3,6,191772.0,90.0,2025-04-06 02:32:42.204402+05,4,152.59,store,EUR,NaN
4,8,112959.0,1827.0,2025-07-15 23:18:42.204402+05,4,190.72,partner,PKR,NaN



Null counts:


notes           899222
channel          34731
customer_id      12889
product_id       10246
sale_ts           5257
sale_id              0
quantity             0
amount               0
currency_raw         0
dtype: int64

Duplicate sale_id count: 40203

Dtypes:
sale_id           int64
customer_id     float64
product_id      float64
sale_ts          object
quantity          int64
amount          float64
channel          object
currency_raw     object
notes            object
dtype: object


In [4]:
# Convert customer_id and product_id to numeric
df["customer_id"] = pd.to_numeric(df["customer_id"], errors="coerce").astype("Int64")
df["product_id"] = pd.to_numeric(df["product_id"], errors="coerce").astype("Int64")

In [5]:
# Drop notes column if it exists, and rename currency_raw to currency if it exists
if "notes" in df.columns:
    df = df.drop(columns=["notes"])

if "currency_raw" in df.columns:
    df = df.rename(columns={"currency_raw": "currency"})

print(df.columns.tolist())

['sale_id', 'customer_id', 'product_id', 'sale_ts', 'quantity', 'amount', 'channel', 'currency']


In [6]:
# Remove duplicate sale_id rows, keeping the first occurrence
before = len(df)
df = df.drop_duplicates(subset=["sale_id"], keep="first")
print("Duplicate sale_id rows removed:", before - len(df))
print("Shape after duplicate removal:", df.shape)

Duplicate sale_id rows removed: 40203
Shape after duplicate removal: (1000000, 8)


In [7]:
# Standardize channel and currency values, and set invalid entries to NaN
valid_channels = ["mobile_app", "online", "partner", "store"]
valid_currencies = ["EUR", "GBP", "PKR", "USD"]

df["channel"] = df["channel"].astype(str).str.strip().str.lower()
df["channel"] = df["channel"].replace({
    "mobile app": "mobile_app",
    "mobile-app": "mobile_app",
    "nan": np.nan,
    "none": np.nan,
    "": np.nan,
})

df["currency"] = df["currency"].astype(str).str.strip().str.upper()
df["currency"] = df["currency"].replace({
    "NAN": np.nan,
    "NONE": np.nan,
    "": np.nan,
})

In [8]:
# Set invalid channels and currencies to NaN
for col in ["customer_id", "product_id", "quantity"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["sale_ts"] = pd.to_datetime(df["sale_ts"], errors="coerce", utc=True)

In [9]:
# Final null handling: drop rows with missing key fields
before = len(df)

df = df.dropna(subset=[
    "customer_id",
    "product_id",
    "sale_ts",
    "quantity",
    "amount",
    "channel",
    "currency",
])

print("Rows removed due to missing key fields:", before - len(df))
print("Shape after null handling:", df.shape)

Rows removed due to missing key fields: 39912
Shape after null handling: (960088, 8)


In [10]:
# For any remaining missing channels, fill with the most common channel
df.loc[~df["channel"].isin(valid_channels), "channel"] = np.nan
channel_mode = df["channel"].mode(dropna=True)[0]
df["channel"] = df["channel"].fillna(channel_mode)

In [11]:
# For any remaining missing currencies, fill with the most common currency
df.loc[~df["currency"].isin(valid_currencies), "currency"] = np.nan
currency_mode = df["currency"].mode(dropna=True)[0]
df["currency"] = df["currency"].fillna(currency_mode)

In [12]:
# Final range filtering based on known valid ranges
min_ts = pd.Timestamp("2025-03-26 19:23:42.204402+05:00")
max_ts = pd.Timestamp("2026-03-26 19:22:42.204402+05:00")
before = len(df)

df = df[
    (df["customer_id"] >= 1) & (df["customer_id"] <= 200000) &
    (df["product_id"] >= 1) & (df["product_id"] <= 20000) &
    (df["quantity"] >= 1) & (df["quantity"] <= 219) &
    (df["amount"] >= 2.04) & (df["amount"] <= 41533.86) &
    (df["sale_ts"] >= min_ts) & (df["sale_ts"] <= max_ts)
].copy()

print("Rows removed due to invalid ranges:", before - len(df))
print("Shape after range filtering:", df.shape)

Rows removed due to invalid ranges: 58286
Shape after range filtering: (901802, 8)


In [13]:
# Save the cleaned dataset
print("Final shape:", df.shape)
print("\nRemaining null counts:")
display(df.isnull().sum())
print("\nRemaining dtypes:")
print(df.dtypes)
display(df.head())

Final shape: (901802, 8)

Remaining null counts:


sale_id        0
customer_id    0
product_id     0
sale_ts        0
quantity       0
amount         0
channel        0
currency       0
dtype: int64


Remaining dtypes:
sale_id                      int64
customer_id                  Int64
product_id                   Int64
sale_ts        datetime64[ns, UTC]
quantity                     int64
amount                     float64
channel                     object
currency                    object
dtype: object


,sale_id,customer_id,product_id,sale_ts,quantity,amount,channel,currency
0,2,93717,10536,2026-02-21 02:55:42.204402+00:00,2,27.84,store,EUR
1,3,140749,4467,2025-05-16 06:12:42.204402+00:00,9,180.83,online,GBP
2,5,1957,1721,2026-01-08 02:52:42.204402+00:00,3,209.91,mobile_app,GBP
3,6,191772,90,2025-04-05 21:32:42.204402+00:00,4,152.59,store,EUR
4,8,112959,1827,2025-07-15 18:18:42.204402+00:00,4,190.72,partner,PKR


In [14]:
# Save the cleaned synthetic dataset
df_synthetic_cleaned = df.copy()
synthetic_output_path = cleaned_data / "corrupted_synthetic_dataset_cleaned.csv"
df_synthetic_cleaned.to_csv(synthetic_output_path, index=False)
print(f"Saved cleaned synthetic data to cleaned data directory.")

Saved cleaned synthetic data to cleaned data directory.


## PakWheels Data Cleaning

In [15]:
# 1. Load CSV
df = pd.read_csv(data / "PakWheelsData.csv")

# 3. Drop assembly
# counting NAN as a unique as well by putting constraint of dropna=False otherwise 1 unique if True
"""print("Unique values in assembly column:", df["assembly"].nunique(dropna=False))
print("\n")
print("Nan vs filled values in assembly column:")
print(df["assembly"].value_counts(dropna=False))
df = df.drop(columns=["assembly"])"""
df["assembly"] = df["assembly"].fillna("Unknown")


# 4. Shape and head
print("\n")
print("Shape:", df.shape)
df.head()



Shape: (62302, 14)


,addref,city,assembly,body,make,model,year,engine,transmission,fuel,color,registered,mileage,price
0,7642697,Islamabad,Unknown,Compact SUV,KIA,Sorento,2021.0,3500.0,Automatic,Petrol,NaN,Islamabad,54654,9300000.0
1,7909608,Sadiqabad,Imported,Hatchback,Daihatsu,Mira,2020.0,660.0,Automatic,Petrol,White,Punjab,10000,3700000.0
2,7929224,Peshawar,Imported,Hatchback,Toyota,Vitz,2018.0,1000.0,Automatic,Petrol,Silver,Islamabad,123000,4150000.0
3,7832366,Lahore,Unknown,Sedan,Toyota,Corolla,2019.0,1600.0,Automatic,Petrol,White,Lahore,105000,4850000.0
4,7866725,Islamabad,Unknown,Sedan,Toyota,Corolla,2022.0,1600.0,Automatic,Petrol,White,Islamabad,6500,6600000.0


In [16]:
# 1. Missing value summary DataFrame
values = df.isnull().sum() 
count = values
percent = (values/len(df))*100
summary = pd.DataFrame()
summary["missing values count"] = count
summary["missing values percentage"] = percent
summary = summary[summary["missing values count"] > 0]
summary = summary.sort_values(by="missing values percentage", ascending=False)
print("Summary of missing values :")
display(summary)

Summary of missing values :


,missing values count,missing values percentage
body,7091,11.381657
year,3786,6.076851
color,1192,1.913261
fuel,709,1.138005
price,472,0.757600
engine,3,0.004815


In [17]:
# 2. Remove outliers based on known valid ranges
df_cleaned = df.copy()
before = len(df_cleaned)

df_cleaned = df_cleaned[
    (df_cleaned["price"]   >= 500_000)    &
    (df_cleaned["price"]   <= 30_000_000) &
    (df_cleaned["mileage"] >= 100)        &
    (df_cleaned["mileage"] <= 500_000)    &
    (df_cleaned["engine"]  >= 600)        &
    (df_cleaned["year"]    >= 1990)
]

print(f"Rows removed: {before - len(df_cleaned):,} | Remaining: {len(df_cleaned):,}")

Rows removed: 6,971 | Remaining: 55,331


In [18]:
# Handle all remaining missing values on df_cleaned.
# Use the strategy you justified in Part 4.
# Remember: rows where 'price' is missing cannot be labelled â€” they must be dropped.

# NOTE: removing outliers results in missing values with only 3 columns so I will be hndlingling only
# those although the strategy discussed in part-4 is the general case strategy 
# tuturial helped: https://www.youtube.com/watch?v=VRmXto2YA2I
# I have debugged this code of groupby part using AI, I acknowledged that but first try is made by myself using tutorial

df_cleaned_2 = df_cleaned.dropna(subset=["body"])
# the lambda function is taking most frequent value after make model body pair and filling body value with it
group_mode = df_cleaned_2.groupby(["make", "model"])["body"].agg(lambda x: x.mode().iloc[0]) 
missing_rows_entries = df_cleaned["body"].isna()

for i in df_cleaned[missing_rows_entries].index:
    make = df_cleaned.loc[i, "make"]
    model = df_cleaned.loc[i, "model"]
    if (make, model) in group_mode.index:
        df_cleaned.loc[i, "body"] = group_mode[(make, model)]

# dropping rows from body after imputation with groupby mode(100% matching values imputed only) the missing%(0.4%) is quite low after that 
# dropping fuel because the missing count is quite low after handling outliers only 0.6%
# filling missing values with unknown in color still closer 2%
df_cleaned = df_cleaned.dropna(subset=["body"])
df_cleaned = df_cleaned.dropna(subset=["fuel"])
df_cleaned["color"] = df_cleaned["color"].fillna("Unknown")

# DO NOT REMOVE â€” must pass before continuing
assert df_cleaned.isnull().sum().sum() == 0, "Still has missing values!"
print("Shape after cleaning:", df_cleaned.shape)
print("No missing values remain")

Shape after cleaning: (54755, 14)
No missing values remain


In [19]:
# Save the cleaned PakWheels dataset
df_pakwheels_cleaned = df_cleaned.copy()
pakwheels_output_path = cleaned_data / "PakWheelsData_cleaned.csv"
df_pakwheels_cleaned.to_csv(pakwheels_output_path, index=False)
print(f"Saved cleaned PakWheels data to cleaned data directory.")

Saved cleaned PakWheels data to cleaned data directory.
